# Charged Particle Motion in a Uniform Magnetic Field

## Physical Background

A charged particle with charge $q$ and mass $m$ moving in a static, uniform magnetic field $\mathbf{B}$ experiences the **Lorentz force**:

$$\mathbf{F} = q\,(\mathbf{v} \times \mathbf{B})$$

The equation of motion is therefore:

$$m\,\frac{d\mathbf{v}}{dt} = q\,(\mathbf{v} \times \mathbf{B})$$

### Key features

- The magnetic force is always **perpendicular** to $\mathbf{v}$, so it does **no work** — kinetic energy is conserved.
- The velocity component **parallel** to $\mathbf{B}$ is unaffected: $v_\parallel = \text{const}$.
- The **perpendicular** component undergoes uniform circular motion at the **cyclotron frequency**:

$$\omega_c = \frac{|q|\,B}{m}$$

- The radius of the circular orbit is the **Larmor radius**:

$$r_L = \frac{m\,v_\perp}{|q|\,B} = \frac{v_\perp}{\omega_c}$$

- Combined parallel and perpendicular motion produces a **helical trajectory** along the field direction.

### Numerical method

We rewrite the second-order ODE as a first-order system for the state vector $\mathbf{y} = (\mathbf{r},\,\mathbf{v})$:

$$\frac{d\mathbf{y}}{dt} = \begin{pmatrix} \mathbf{v} \\ \frac{q}{m}(\mathbf{v} \times \mathbf{B}) \end{pmatrix}$$

This is integrated using the **4th-order Runge–Kutta** method (RK45 adaptive scheme from `scipy.integrate.solve_ivp`).

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

## 1. Parameters and Initial Conditions

In [ ]:
# Physical constants (proton in a 0.1 T field)
q  = 1.602e-19       # charge  [C]
m  = 1.673e-27       # mass    [kg]
B0 = 0.1             # field magnitude [T]
B_hat = np.array([0., 0., 1.])   # B direction (z-axis)

v_perp = 1e5   # perpendicular speed [m/s]
v_par  = 3e4   # parallel speed      [m/s]

r0 = np.array([0., 0., 0.])
v0 = np.array([v_perp, 0., v_par])
y0 = np.concatenate([r0, v0])

omega_c = abs(q) * B0 / m
T_c = 2 * np.pi / omega_c
r_L = v_perp / omega_c

print(f"Cyclotron frequency  ω_c = {omega_c:.4e} rad/s")
print(f"Cyclotron period     T_c = {T_c:.4e} s")
print(f"Larmor radius        r_L = {r_L:.4e} m")

## 2. Equation of Motion and Integration

In [ ]:
def lorentz(t, y):
    """dy/dt = [v, (q/m)(v × B)]"""
    v = y[3:6]
    B = B0 * B_hat
    a = (q / m) * np.cross(v, B)
    return np.concatenate([v, a])

t_span = (0, 5 * T_c)
t_eval = np.linspace(*t_span, 2000)

sol = solve_ivp(lorentz, t_span, y0, method="RK45",
                t_eval=t_eval, rtol=1e-10, atol=1e-12)

x, y_coord, z = sol.y[0], sol.y[1], sol.y[2]
vx, vy, vz    = sol.y[3], sol.y[4], sol.y[5]

print(f"Solver success: {sol.success}")
print(f"Integration points: {sol.t.size}")

## 3. 3D Helical Trajectory

In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

ax.plot(x, y_coord, z, linewidth=0.8, color="#2563eb")
ax.scatter(*r0, color="red", s=40, zorder=5, label="Start")

ax.set_xlabel("x  [m]")
ax.set_ylabel("y  [m]")
ax.set_zlabel("z  [m]")
ax.set_title("Proton trajectory in uniform B = 0.1 T ẑ")
ax.legend()
plt.tight_layout()
plt.savefig("trajectory_3d.png", dpi=150)
plt.show()

## 4. Projection onto the xy-plane (Circular Motion)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(x, y_coord, linewidth=0.7, color="#2563eb")
ax.scatter(*r0[:2], color="red", s=40, zorder=5, label="Start")
ax.set_xlabel("x  [m]")
ax.set_ylabel("y  [m]")
ax.set_title("xy-plane projection (Larmor orbit)")
ax.set_aspect("equal")
ax.legend()
plt.tight_layout()
plt.savefig("projection_xy.png", dpi=150)
plt.show()

## 5. Energy Conservation Check

Since $\mathbf{F} \perp \mathbf{v}$, the kinetic energy must be exactly conserved.  
Any deviation is a measure of the numerical error.

In [ ]:
KE = 0.5 * m * (vx**2 + vy**2 + vz**2)
KE_rel = (KE - KE[0]) / KE[0]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(sol.t / T_c, KE_rel, linewidth=0.8, color="#dc2626")
ax.set_xlabel("t / T_c")
ax.set_ylabel("ΔE / E₀")
ax.set_title("Relative kinetic-energy error")
ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 2))
plt.tight_layout()
plt.savefig("energy_conservation.png", dpi=150)
plt.show()

print(f"Max |ΔE/E₀| = {np.max(np.abs(KE_rel)):.2e}")

## 6. Drift in a Non-Uniform Field (E × B Drift)

Adding a uniform electric field $\mathbf{E}$ perpendicular to $\mathbf{B}$ produces the well-known **E × B drift**:

$$\mathbf{v}_D = \frac{\mathbf{E} \times \mathbf{B}}{B^2}$$

The guiding centre drifts in the direction perpendicular to both $\mathbf{E}$ and $\mathbf{B}$, independent of the particle's charge and mass.

In [ ]:
E_field = np.array([0., 500., 0.])   # E in y-direction [V/m]

def lorentz_EB(t, y):
    """Lorentz force with both E and B fields."""
    v = y[3:6]
    B = B0 * B_hat
    a = (q / m) * (E_field + np.cross(v, B))
    return np.concatenate([v, a])

sol_EB = solve_ivp(lorentz_EB, t_span, y0, method="RK45",
                   t_eval=t_eval, rtol=1e-10, atol=1e-12)

# Analytical drift velocity
v_drift = np.cross(E_field, B0 * B_hat) / B0**2
print(f"E×B drift velocity = {v_drift} m/s")

In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

ax.plot(sol_EB.y[0], sol_EB.y[1], sol_EB.y[2],
        linewidth=0.8, color="#7c3aed", label="Trajectory")
ax.scatter(*r0, color="red", s=40, zorder=5, label="Start")

ax.set_xlabel("x  [m]")
ax.set_ylabel("y  [m]")
ax.set_zlabel("z  [m]")
ax.set_title("Proton trajectory with E×B drift\n(E = 500 V/m ŷ, B = 0.1 T ẑ)")
ax.legend()
plt.tight_layout()
plt.savefig("trajectory_ExB_drift.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sol_EB.y[0], sol_EB.y[1], linewidth=0.7, color="#7c3aed")
ax.scatter(*r0[:2], color="red", s=40, zorder=5, label="Start")
ax.set_xlabel("x  [m]")
ax.set_ylabel("y  [m]")
ax.set_title("xy-projection with E×B drift")
ax.set_aspect("equal")
ax.legend()
plt.tight_layout()
plt.savefig("projection_ExB_xy.png", dpi=150)
plt.show()

## Summary

| Scenario | Trajectory | Key physics |
|----------|-----------|-------------|
| Pure B-field | Helix along ẑ | Cyclotron motion + free streaming |
| E ⊥ B | Drifting helix | E×B drift perpendicular to both fields |

The RK45 integrator conserves energy to machine precision (~10⁻¹² relative error), confirming numerical accuracy.